# Bivariate-Poisson NHL goals model — Colab tier

Fits one generative score model (attack/defense/home-adv + Dixon-Coles low-score
term) and derives moneyline, totals, and puck-line *consistently* from the joint
score distribution. See `models/poisson_nhl.py`.

Flow: export NHL game features locally, fit here, download `nhl_poisson.pkl` into
`data/exports/artifacts/`, then `python run.py --import-models`.


In [ ]:
!pip -q install scipy polars pyarrow


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; BASE='/content/drive/MyDrive/sports-edge'
EXPORTS=BASE+'/exports'; ARTIFACTS=BASE+'/artifacts'; os.makedirs(ARTIFACTS,exist_ok=True)


In [ ]:
import sys; sys.path.append('/content/sports-edge')  # clone the repo here first
import polars as pl, models.poisson_nhl as pn
df=pl.read_parquet(EXPORTS+'/nhl_game_features.parquet').filter(pl.col('is_home')=='H' )
# (use raw nhl_team_games if the export lacks home rows; both carry goals/opp_goals)
params=pn.fit(df['team_id'],df['opp_team_id'],df['goals'],df['opp_goals'])
print('home_adv',round(params['home_adv'],3),'rho',round(params['rho'],3),'teams',len(params['teams']))
import pickle; pickle.dump(params, open(ARTIFACTS+'/nhl_poisson.pkl','wb'))
print('wrote nhl_poisson.pkl')


`import_models` already globs `nhl_*`, so `nhl_poisson.pkl` comes across with
everything else. Serve it via `models.poisson_nhl.load()` + `predict_game()` —
every market falls out of the one fitted grid, no scipy needed locally.
